# NCRB scanned tables: cloud OCR

Reads every scanned page of NCRB's old reports (about 11,700 pages, mostly before 2000) with a current document-OCR model on a Colab GPU, and saves each page's output to your Google Drive in the layout the [NCRB pipeline](https://github.com/reclaimchennai/NCRB) reads.

1. **Runtime → Change runtime type → GPU.** An **A100** or **L4** is best (Colab Pro / pay-as-you-go); a free **T4** works, about 3–5× slower.
2. Run the cells in order. Step 4 is a short bake-off: three models read the same ~150 pages, and the one whose tables best match NCRB's own printed row and column totals wins. Step 5 runs the winner on everything.
3. If the session ends, run steps 1–3 again and re-run step 5: pages already on Drive are skipped.
4. When it finishes, download `ncrb_ocr/<model>.zip` from Drive and import it on the laptop (step 6).

Pages and the manifest come from the [`ocr-pages` release](https://github.com/reclaimchennai/NCRB/releases/tag/ocr-pages). Nothing is uploaded anywhere except your own Drive.

## 1. GPU, Drive and code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/NCRB && git -C /content/NCRB pull -q || git clone -q --depth 1 https://github.com/reclaimchennai/NCRB.git /content/NCRB
!mkdir -p /content/drive/MyDrive/ncrb_ocr/ocr_cache

## 2. Install vLLM (3–5 minutes)

In [ ]:
!pip install -q -U vllm pymupdf requests beautifulsoup4 lxml
import torch
DTYPE = 'auto' if torch.cuda.get_device_capability()[0] >= 8 else 'half'   # T4 has no bfloat16
print(torch.cuda.get_device_name(), '| dtype', DTYPE)

## 3. The scanned pages (about 2–3 GB)

In [ ]:
import json, subprocess, os
REL = 'https://github.com/reclaimchennai/NCRB/releases/download/ocr-pages'
os.makedirs('/content/ocr_pages', exist_ok=True)
!curl -sL -o /content/ocr_pages/manifest.json {REL}/manifest.json
man = json.load(open('/content/ocr_pages/manifest.json'))
for part in man['parts']:
    if not os.path.exists(f'/content/ocr_pages/{part}.done'):
        subprocess.run(f'curl -sL {REL}/{part} | tar -x -C /content/ocr_pages && touch /content/ocr_pages/{part}.done', shell=True, check=True)
files = man['files']
print(len(files), 'files,', sum(len(f['pages']) for f in files), 'scanned pages;', sum(len(f['pages']) for f in files if f['bench']), 'in the bake-off')

## 4. Bake-off on ~150 pages (about 20–40 minutes)

Each model runs in its own process so the GPU is freed between them. The score counts NCRB's printed totals that the model's table reproduces; a missed total costs twice a matched one. Benchmarks on modern documents (OmniDocBench v1.6, 2026) rank TeleOCR first for tables, then PaddleOCR-VL-1.6, then GLM-OCR; old typewritten scans can differ, which is why we test.

In [ ]:
CANDIDATES = ['XingChen-AGI/TeleOCR', 'PaddlePaddle/PaddleOCR-VL-1.6', 'zai-org/GLM-OCR']
for m in CANDIDATES:
    !cd /content/NCRB && python colab/ocr_pages.py --model {m} --bench --dtype {DTYPE}
!cd /content/NCRB && python colab/score.py --bench

## 5. Every scanned page with the winner (A100: a few hours; resumable)

In [ ]:
board = json.load(open('/content/drive/MyDrive/ncrb_ocr/ocr_cache/leaderboard.json'))
SLUG = {m.split('/')[-1]: m for m in CANDIDATES}
WINNER = SLUG.get(board[0]['model'], CANDIDATES[0])   # or set by hand, e.g. WINNER = 'XingChen-AGI/TeleOCR'
print('running', WINNER)
!cd /content/NCRB && python colab/ocr_pages.py --model {WINNER} --dtype {DTYPE}

## 6. Pack the result for the laptop

In [ ]:
slug = WINNER.split('/')[-1]
!cd /content/drive/MyDrive/ncrb_ocr/ocr_cache && zip -qr ../{slug}.zip {slug} leaderboard.json && ls -lh ../{slug}.zip
print(f'Download ncrb_ocr/{slug}.zip from Drive, then on the laptop:')
print(f'  uv run python -m ncrb.vlm_import ~/Downloads/{slug}.zip')